# zero-grad-set-none — faded example 3: Backward reallocates after None

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `zero-grad-set-none`. Running the beacon reports progress on the `PyTorch: zero_grad` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: zero_grad` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`zero-grad-set-none`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "zero-grad-set-none"
DD_SUBTOPIC = "PyTorch: zero_grad"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

After `.grad = None`, the next `.backward()` allocates a fresh gradient tensor — None behaves as zero and does not corrupt the next accumulation. This is why setting None is equivalent to zeroing the buffer.

## Faded exercise 3

### Clear then reaccumulate

Implement `clear_then_backward(w)`: given a leaf tensor `w` that already has a grad, clear it with the None convention, then run `(w * w).sum().backward()` so a fresh grad is allocated, and return the new grad. Complete the clearing statement.

**Fill in:** setting w.grad = None to clear before the next backward

In [ ]:
import torch as t

def clear_then_backward(w):
    w.grad = t.zeros_like(w)  # TODO: set w.grad = None to truly clear it before the next backward
    grad_was_none = w.grad is None  # True only if grad was set to None, not a zeros buffer
    (w * w).sum().backward()
    return w.grad, grad_was_none

w = t.tensor([3.0, 4.0], requires_grad=True)
(w.sum()).backward()
print(clear_then_backward(w))

def _test():
    w = t.tensor([3.0, 4.0], requires_grad=True)
    # seed an existing grad of ones
    (w.sum()).backward()
    assert t.allclose(w.grad, t.ones_like(w))
    out, grad_was_none = clear_then_backward(w)
    # the clear must produce a genuine None, not a zeros buffer
    assert grad_was_none is True, 'grad must be set to None, not a zeros tensor'
    # independent truth: d(sum(w^2))/dw = 2w, computed FRESH (not added to the old ones)
    assert t.allclose(out, 2 * w.detach()), out
    # specifically NOT 2w + 1 (which would mean the old grad was not cleared)
    assert not t.allclose(out, 2 * w.detach() + 1)

try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def clear_then_backward(w):
    w.grad = None
    grad_was_none = w.grad is None  # True only if grad was set to None, not a zeros buffer
    (w * w).sum().backward()
    return w.grad, grad_was_none

w = t.tensor([3.0, 4.0], requires_grad=True)
(w.sum()).backward()
print(clear_then_backward(w))
```
</details>